# Import

In [95]:
import unicodedata
import re
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from torch.utils.tensorboard import SummaryWriter
from nltk.translate.bleu_score import sentence_bleu
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.10.8
numpy 1.26.4
pandas 2.3.3
sklearn 1.8.0
torch 2.9.1+cpu
cpu


# 0. Prepare Data

In [96]:
# 因为西班牙语有一些是特殊字符，所以我们需要unicode转ascii
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

In [97]:
def preprocess_sentence(s):
    """
    在单词与跟在其后的标点符号之间插入一个空格  eg: "he is a boy." => "he is a boy . "
    """
    
    s = unicode_to_ascii(s.lower().strip())     # 变为小写，去掉多余的空格
    s = re.sub(r"([?.!,¿])", r" \1 ", s)         # 在标点符号（?.!,¿）前后添加空格
    s = re.sub(r'[" "]+', " ", s)                # 将多个连续空格替换为单个空格
    s = re.sub(r"[^a-zA-Z?.!,¿]+", " ", s)      # 将不在允许字符集中的所有字符替换为空格
    # s = re.sub(r'[" "]+', " ", s)                # 再次合并可能的连续空格
    
    return s.rstrip().strip()

In [98]:
class LangPairDataset(Dataset):
    """
    双语平行语料数据集类，用于机器翻译任务
    处理西班牙语-英语平行语料，支持训练集/测试集划分和缓存机制
    """
    
    fpath = Path(r"./data_spa_en/spa.txt")          # 数据文件路径
    cache_path = Path(r"./.cache/lang_pair.npy")    # 缓存文件路径
    # 按照9:1划分训练集和测试集
    split_index = np.random.choice(a=['train', 'test'], replace=True, p=[0.9, 0.1], size=118964)
    
    def __init__(self, mode = "train", cache = False):
        if not cache or not self.cache_path.exists():
            # 如果没有缓存，或者缓存不存在，就处理一下数据
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)   #创建缓存文件夹，如果存在就忽略
            
            with open(self.fpath, "r", encoding='utf-8') as file:
                lines = file.readlines()
                lang_pair = [[preprocess_sentence(word) for word in line.split('\t')] 
                             for line in lines]    # 处理数据，变成list((trg, src))的形式
                trg, src = zip(*lang_pair)  # 分离出目标语言和源语言
                trg = np.array(trg)         # 转换为numpy数组
                src = np.array(src)
                
                np.save(self.cache_path, {"trg": trg, "src": src})  #保存为npy文件,方便下次直接读取,不用再处理
        else:
            # 存在缓存，直接读取
            # 读取npy文件，allow_pickle=True允许读取字典
            lang_pair = np.load(self.cache_path, allow_pickle=True).item()  
            trg = lang_pair["trg"]
            src = lang_pair['src']
            
        # self.split_index == mode 返回布尔数组，用于选择对应模式的数据     
        self.trg = trg[self.split_index == mode]    # 按照split_index拿到训练集的标签语言 --英语
        self.src = src[self.split_index == mode]    # 按照split_index拿到训练集的源语言 --西班牙
        
    def __getitem__(self, index):
        return self.trg[index], self.src[index]
    
    def __len__(self):
        return len(self.src)

In [99]:
def get_word_idx(dataset, mode = "src", threshold = 1):
    """
    构建词汇表（word2idx）和反向词汇表（idx2word）
    词汇表就像一本"单词-索引"字典，将文本中的每个单词映射到一个唯一的整数ID，
    这是神经网络处理文本数据的前提（神经网络只能处理数值，不能直接处理文本）

    Args:
        dataset: LangPairDataset对象，包含源语言和目标语言的句子对
        mode: 选择构建哪种语言的词汇表
              - "src": 源语言（西班牙语）
              - "trg": 目标语言（英语）
        threshold: 词频阈值，出现次数低于此值的单词会被舍弃（标记为[UNK]）

    Returns:
        word2idx: 字典，单词 -> 索引
        idx2word: 字典，索引 -> 单词
    """

    # 初始化词汇表，包含4个特殊token
    # 这些特殊token用于处理序列数据的边界情况和特殊需求
    word2idx = {
        "[PAD]": 0,     # 填充 token
        "[BOS]": 1,     # begin of sentence
        "[UNK]": 2,     # 未知 token
        "[EOS]": 3,     # end of sentence
    }
    idx2word = {idx:word for word, idx in word2idx.items()}
    index = len(idx2word)

    # 从数据集中提取所有句子并分词
    # pair[0]是目标语言（英语），pair[1]是源语言（西班牙语）
    # 如果mode='trg'选择pair[0]，否则选择pair[1]
    # 将所有句子连接成一个长字符串，然后按空格分词
    word_list = " ".join([pair[0 if mode=='trg' else 1] for pair in dataset]).split()

    counter = Counter(word_list)    # 统计词频,counter类似字典，key是单词，value是出现次数

    print(f"原始词汇量（去重后）: {len(counter)}")

    for token, count in counter.items():
        if count >= threshold:
            # 出现次数大于阈值的token加入词表
            word2idx[token] = index
            idx2word[index] = token
            index += 1


    print(f"最终词汇量（词频)≥{threshold}）: {len(word2idx) - 4}")

    return word2idx, idx2word

In [100]:
train_ds = LangPairDataset(mode = "train", cache = True)
print(len(train_ds))
src_word2idx, src_idx2word = get_word_idx(train_ds, "src") #源语言词表
trg_word2idx, trg_idx2word = get_word_idx(train_ds, "trg") #目标语言词表

107261
原始词汇量（去重后）: 23774
最终词汇量（词频)≥1）: 23774
原始词汇量（去重后）: 12465
最终词汇量（词频)≥1）: 12465


In [ ]:
class Tokenizer:
    """
    文本分词器类：负责文本和索引序列之间的相互转换,这是连接原始文本和神经网络模型的桥梁
    
    主要功能：
    1. encode: 将文本句子转换为模型可处理的索引序列（添加特殊token、padding等）
    2. decode: 将模型输出的索引序列转换回可读的文本句子
    """
    def __init__(self, word2idx, idx2word, 
                 max_length = 500, pad_idx = 0, bos_idx = 1, unk_idx = 2, eos_idx = 3):
        self.word2idx = word2idx
        self.idx2word = idx2word
        self.max_length = max_length
        self.pad_idx = pad_idx
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.unk_idx = unk_idx
        
    def encode(self, text_list, 
               padding_first = False, add_bos = True, add_eos = True, return_mask = False):
        """
        将文本句子列表编码为索引序列
        
        Args:
            text_list: 文本句子列表，每个句子是空格分隔的单词字符串
            padding_first: padding位置，True表示在序列开头填充，False表示在序列末尾填充
            add_bos: 是否在序列开头添加[BOS]标记
            add_eos: 是否在序列末尾添加[EOS]标记
            return_mask: 是否返回padding mask（用于忽略padding位置的loss计算）
            
        Returns:
            如果return_mask为True: (indices, mask)
                - indices: 形状为(batch_size, seq_len)的索引张量
                - mask: 形状为(batch_size, seq_len)的mask张量，padding位置为1，其余为0
            如果return_mask为False: 只返回indices张量
        """
        
        # 计算实际的最大长度：不能超过预设的max_length，同时考虑特殊token的数量
        max_length = min(self.max_length, add_bos + add_eos + max([len(text) for text in text_list]))
        
        idx_list = []   # 将文本句子列表编码为索引序列
        
        for text in text_list:
            idx = [self.word2idx.get(word, self.unk_idx) for word in text[:max_length - add_eos - add_bos]]     # 截断到最大长度（预留特殊token(eos, bos)的位置）
            if add_bos:
                idx = [self.bos_idx] + idx
            if add_eos:
                idx += [self.eos_idx]
            if padding_first:
                idx = [self.pad_idx] * (max_length - len(idx)) + idx
            else:
                idx =  idx + [self.pad_idx] * (max_length - len(idx))
                
            idx_list.append(idx)
        
        idx_list = torch.tensor(idx_list)   #转换为tensor
        masks = (idx_list == self.pad_idx).to(dtype=torch.int64)    # mask用于去除padding对loss的影响
        
        return (idx_list, masks) if return_mask else idx_list
        
    def decode(self, idx_list, 
               remove_bos = True, remove_eos = True, remove_pad = True, padding_first = False, split = False):
        """
        将索引序列解码回文本句子
        
        Args:
            idx_list: 索引序列，可以是列表、numpy数组或PyTorch张量
            remove_bos: 是否移除[BOS]标记
            remove_eos: 是否移除[EOS]标记（遇到[EOS]停止解码）
            remove_pad: 是否移除[PAD]标记（遇到[PAD]停止解码）
            split: 返回格式，True返回单词列表，False返回字符串
            
        Returns:
            解码后的文本句子列表
        """
        text_list = []      # 解码后的文本句子列表
        
        for indices in idx_list:
            text = []
            for idx in indices:
                word = self.idx2word.get(idx, "[UNK]")      # 将单个索引转换为单词
                # 根据设置处理特殊token
                if remove_bos and word == "[BOS]":
                    continue
                if remove_eos and word == "[EOS]":
                    break
                if remove_pad and word == "[PAD]" and padding_first == True:    
                    continue
                if remove_pad and word == "[PAD]":
                    break
                text.append(word)
                
            # 根据split参数决定返回格式    
            text_list.append(" ".join(text) if not split else text)
            
        return text_list

In [ ]:
def collate_fct(batch):
    """
    自定义的批处理函数，用于将一批数据样本整理成模型训练所需的格式。
    
    该函数的主要作用是：
    1. 将输入的批量数据（每个样本包含源语言和目标语言的句子对）进行分词处理；
    2. 对源语言和目标语言分别进行编码，生成模型输入所需的张量；
    3. 添加特殊标记（如[BOS]、[EOS]、[PAD]）并生成相应的掩码；
    4. 返回一个字典，包含编码后的输入张量及其对应的掩码。

    Args:
        batch (list): 一批数据样本，每个样本是一个元组(pair)，其中：
                      pair[0]为目标语言句子（字符串）
                      pair[1]为源语言句子（字符串）

    Returns:
        dict: 包含以下键值对的字典：
              - "encoder_inputs": 源语言句子编码后的输入张量（带[BOS]和[EOS]）
              - "encoder_inputs_mask": 源语言输入的掩码张量（用于屏蔽填充部分）
              - "decoder_inputs": 目标语言句子编码后的解码器输入张量（带[BOS]，不带[EOS]）
              - "decoder_labels": 目标语言句子编码后的标签张量（带[EOS]，不带[BOS]）
              - "decoder_labels_mask": 目标语言标签的掩码张量（用于屏蔽填充部分）
    """
    
    # 提取目标语言和源语言的单词列表
    trg_words = [pair[0].split() for pair in batch]
    src_words = [pair[1].split() for pair in batch]
    
    # 编码源语言输入：[PAD] [BOS] src [EOS]，并生成掩码
    encoder_inputs, encoder_inputs_mask = src_tokenizer.encode(src_words, 
                                                               padding_first = True,
                                                               add_bos = True,
                                                               add_eos = True,
                                                               return_mask = True)
    
    # 编码目标语言解码器输入：[BOS] trg [PAD]
    decoder_inputs = trg_tokenizer.encode(trg_words, 
                                        padding_first = False,
                                        add_bos = True,
                                        add_eos = False,
                                        return_mask = False)
    
    # 编码目标语言标签：trg [EOS] [PAD]，并生成掩码
    decoder_labels, decoder_labels_mask = trg_tokenizer.encode(trg_words,
                                                               padding_first = False,
                                                               add_bos = False,
                                                               add_eos = True,
                                                               return_mask = True)
    
    return {
        "encoder_inputs": encoder_inputs.to(device=device),
        "encoder_inputs_mask": encoder_inputs_mask.to(device=device),
        "decoder_inputs": decoder_inputs.to(device=device),
        "decoder_labels": decoder_labels.to(device=device),
        "decoder_labels_mask": decoder_labels_mask.to(device=device),
        } #当返回的数据较多时，用dict返回比较合理 


# 1. Define Model

In [ ]:
class Encoder(nn.Module):
    """
    编码器（Encoder）类：将输入序列编码为隐藏状态
    在Seq2Seq模型中，编码器负责"理解"输入句子，将其信息压缩为
    一个上下文向量（hidden state），供解码器生成目标句子

    主要功能：
    1. 接收源语言的词嵌入序列作为输入；
    2. 输出编码后的隐藏状态和最终的上下文向量，供解码器使用。

    Args:
        vocab_size (int): 输入词嵌入的维度（即词汇表大小）。
        embedding_dim (int) : 词嵌入维度
        hidden_size (int): GRU 隐藏层的维度。
        num_layers (int): GRU 的层数，默认为 1。
    """
    
    def __init__(self, vocab_size, embedding_dim = 256, hidden_dim = 1024, num_layers = 1):
        super().__init__()
        # 词嵌入层：将离散的词索引转换为密集的词向量
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # GRU层：循环神经网络的一种，处理序列数据
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        
    def forward(self, encoder_inputs):   # encoder_inputs.shape = [batch size, sequence length]
        
        embeds = self.embedding(encoder_inputs)     # [batch_size, seq_len] -> [batch_size, seq_len, embedding_dim]
        
        seq_output, last_hidden = self.gru(embeds)       #   seq_output: [batch_size, seq_len, hidden_dim]  (所有时间步的输出)
                                                         #   last_hidden: [num_layers, batch_size, hidden_dim]   (最后一个时间步的隐藏状态)
        return seq_output, last_hidden
    

In [ ]:
class BahdanauAttention(nn.Module):
    """
    Bahdanau注意力机制（加法注意力）
    
    在Seq2Seq模型中，注意力机制允许解码器在生成每个词时，"关注"编码器输出的不同部分。
    
    核心思想：不是依赖单一的上下文向量，而是为每个解码时间步动态计算上下文向量
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # Wk: 对keys（编码器所有时间步的输出）进行线性变换
        # 输入/输出形状: [batch_size, seq_len, hidden_dim] -> [batch_size, seq_len, hidden_dim]
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        
        # Wq: 对query（解码器当前隐藏状态）进行线性变换
        # 输入形状: [batch_size, hidden_dim] -> 输出形状: [batch_size, hidden_dim]
        self.Wq = nn.Linear(hidden_dim, hidden_dim) 
        
        # V: 将变换后的query和keys的组合映射为一个标量分数
        # 输入形状: [batch_size, seq_len, hidden_dim] -> 输出形状: [batch_size, seq_len, 1]
        self.V = nn.Linear(hidden_dim, 1)
        
    def forward(self, query, keys, values, attn_mask = None):
        """
        计算注意力上下文向量
        
        Args:
            query: 解码器的当前隐藏状态，形状 [batch_size, hidden_dim]
                  这是当前解码时间步要"提问"的向量，想知道编码器的哪些部分与当前生成相关
                  
            keys: 编码器所有时间步的输出，形状 [batch_size, seq_len, hidden_dim]
                  这是用于计算注意力分数的"键"，决定query与每个编码器位置的相关性
                  
            values: 编码器所有时间步的输出，形状 [batch_size, seq_len, hidden_dim]
                   这是实际要"关注"的内容，通常与keys相同，但在某些变体中可能不同
                   
            attn_mask: 可选的注意力掩码，形状 [batch_size, seq_len]
                      用于屏蔽padding位置的注意力（1表示padding位置）
        
        Returns:
            context_vector: 上下文向量，形状 [batch_size, hidden_dim]
                           根据注意力权重加权求和得到的向量，包含当前解码步需要的输入信息
                           
            scores: 注意力权重，形状 [batch_size, seq_len, 1]
                   每个编码器位置的重要性权重，总和为1
        """
        query = query.unsqueeze(-2)         # query.shape = [batch size, hidden_dim] -->通过unsqueeze(-2)增加维度 [batch size, 1, hidden_dim]
        
        # 计算注意力分数
        # Bahdanau注意力的核心公式: e = V * tanh(Wk * keys + Wq * query) 
        # 计算过程：
        # 1. self.Wk(keys): 对keys进行线性变换 [batch, seq_len, hidden]
        # 2. self.Wq(query): 对query进行线性变换 [batch, 1, hidden]
        # 3. 相加: 通过广播机制，[batch, seq_len, hidden] + [batch, 1, hidden] = [batch, seq_len, hidden]
        # 4. tanh: 非线性激活
        # 5. self.V: 线性变换降维到1，得到每个位置的标量分数 [batch, seq_len, 1]
        scores = self.V(F.tanh(self.Wk(keys) + self.Wq(query)))
        
        if attn_mask is not None:
            # attn_mask: [batch_size, seq_len]，1表示padding位置
            attn_mask = (attn_mask.unsqueeze(-1)) * -1e16       # 将padding位置的分数设为非常大的负数
            scores += attn_mask                                 # 加上负无穷，softmax后这些位置的概率接近0
            
        scores = F.softmax(scores, dim=-2)       # 在seq_len维度上做softmax，使得每个批次的所有位置权重和为1
        
        # 计算上下文向量（注意力加权求和）
        # values: [batch_size, seq_len, hidden_dim]
        # scores: [batch_size, seq_len, 1]
        # 逐元素相乘: [batch_size, seq_len, hidden_dim] * [batch_size, seq_len, 1] -> [batch_size, seq_len, hidden_dim]
        # 在seq_len维度求和: [batch_size, hidden_dim]
        context_vector = torch.mul(scores, values).sum(dim=-2)
        
        return context_vector, scores

In [ ]:
class Decoder(nn.Module):
    """
    解码器（Decoder）类：在注意力机制的辅助下，逐步生成目标语言序列
    在Seq2Seq模型中，解码器负责根据编码器输出的上下文信息和已生成的目标词，
    逐个预测下一个目标词。每一步都会使用注意力机制关注输入序列的相关部分。
    """
    def __init__(self, vocab_size, embedding_dim = 256, hidden_dim = 1024, num_layers = 1):
        """
        初始化解码器
        
        Args:
            vocab_size: 目标语言词汇表大小
            embedding_dim: 词嵌入维度
            hidden_dim: GRU隐藏层维度
            num_layers: GRU层数
        """
        super().__init__()
        # 词嵌入层：将目标词索引转换为稠密向量
        # 输入: [batch_size, 1] -> 输出: [batch_size, 1, embedding_dim]
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # GRU层：处理序列数据，生成隐藏状态
        # 输入维度: embedding_dim + hidden_dim (词嵌入 + 上下文向量)
        # 输出维度: hidden_dim
        # 为什么是embedding_dim + hidden_dim？因为每个时间步的输入是词嵌入和上下文向量的拼接
        self.gru = nn.GRU(embedding_dim + hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        
        self.final_classifer = nn.Linear(hidden_dim, vocab_size)    # 最终分类器：将GRU输出映射到词汇表大小的logits, 用于计算每个目标词的概率分布
        self.dropout = nn.Dropout(0.6)                              # Dropout层：防止过拟合
        self.attention = BahdanauAttention(hidden_dim)              # 注意力机制：帮助解码器关注输入序列的相关部分
        
    def forward(self, decoder_inputs, hidden, encoder_outputs, attn_mask = None):
        """
        单步解码前向传播
        
        Args:
            decoder_inputs: 当前时间步的输入词索引，形状 [batch_size, 1],通常是上一个时间步预测的词，或教师强制时的真实词
                          
            hidden: *解码器的隐藏状态，形状 [batch_size, hidden_dim],第一次调用时使用编码器的最终隐藏状态,后续使用上一个时间步的隐藏状态
                   
            encoder_outputs: 编码器所有时间步的输出，形状 [batch_size, seq_len, hidden_dim],用于注意力机制计算
                            
            attn_mask: 注意力掩码，形状 [batch_size, seq_len],用于屏蔽编码器输出的padding位置,1表示padding位置，这些位置不参与注意力计算
        
        Returns:
            logits: 预测logits，形状 [batch_size, 1, vocab_size],经过softmax后可得到下一个词的概率分布
                   
            hidden: 更新后的隐藏状态，形状 [batch_size, hidden_dim],用于下一个时间步的解码
                   
            attention_score: 注意力权重，形状 [batch_size, seq_len, 1],显示解码器在当前时间步关注了输入序列的哪些部分
        """
        # decoder_input.shape = [batch size, 1]
        assert len(decoder_inputs.shape) == 2 and decoder_inputs.shape[-1] == 1, f"decoder_input.shape = {decoder_inputs.shape}"
        # hidden.shape = [batch size, hidden_dim]，decoder_hidden,而第一次使用的是encoder的hidden
        assert len(hidden.shape) == 2, f"hidden.shape = {hidden.shape}"
        # encoder_outputs.shape = [batch size, sequence length, hidden_dim]
        assert len(encoder_outputs.shape) == 3 ,f"encoder_outputs.shape = {encoder_outputs.shape}"
        
        # ============ 步骤1：计算注意力上下文向量 ============
        # 使用当前隐藏状态作为query，编码器输出作为keys/values
        # 注意力机制会计算当前解码状态与每个编码器位置的相关性
        # context_vector: [batch_size, hidden_dim] - 包含输入序列中与当前解码相关的信息
        # attention_score: [batch_size, seq_len, 1] - 注意力权重分布
        context_vector, attention_score = self.attention(query=hidden, keys=encoder_outputs, values=encoder_outputs, attn_mask=attn_mask)   
        # context_vector.shape = [batch size, hidden_dim]
        
        # ============ 步骤2：词嵌入 ============
        # 将当前输入的词索引转换为向量表示
        # [batch_size, 1] -> [batch_size, 1, embedding_dim]
        embeds = self.embedding(decoder_inputs)
        
        # ============ 步骤3：拼接上下文向量和词嵌入 ============
        # 将上下文向量扩展一维，然后与词嵌入拼接
        # context_vector: [batch_size, hidden_dim] -> unsqueeze(-2) -> [batch_size, 1, hidden_dim]
        # embeds: [batch_size, 1, embedding_dim]
        # 拼接后: [batch_size, 1, embedding_dim + hidden_dim]
        embeds = torch.cat([context_vector.unsqueeze(-2), embeds], dim = -1)
        
        # ============ 步骤4：GRU前向传播 ============
        # 输入当前步的拼接向量，更新隐藏状态
        # seq_output: [batch_size, 1, hidden_dim] - 当前步的GRU输出
        # hidden: [batch_size, hidden_dim] - 更新后的隐藏状态（用于下一步）
        seq_output, hidden = self.gru(embeds)
        
        # ============ 步骤5：生成预测logits ============
        # 将GRU输出通过分类器得到词汇表大小的logits
        # 先应用dropout防止过拟合，再线性变换
        # [batch_size, 1, hidden_dim] -> [batch_size, 1, vocab_size]
        logits = self.final_classifer(self.dropout(seq_output))
        
        return logits, hidden, attention_score
        

In [ ]:
class Sequence2Sequence(nn.Module):
    """
    序列到序列（Seq2Seq）模型：完整的编码器-解码器架构，支持注意力机制
    
    这是机器翻译的核心模型，包含：
    1. 编码器（Encoder）：将源语言句子编码为隐藏状态序列
    2. 解码器（Decoder）：在注意力机制辅助下，逐步生成目标语言句子
    3. 支持训练模式（forward）和推理模式（infer）
    """
    def __init__(self,
                trg_vocab_size,                 # 目标语言词汇表大小
                src_vocab_size,                 # 源语言词汇表大小
                encoder_embedding_dim = 256,    # 编码器词嵌入维度
                encoder_hidden_dim = 1024,      # 编码器隐藏层维度
                encoder_num_layer = 1,          # 编码器GRU层数
                decoder_embedding_dim = 256,    # 解码器词嵌入维度
                decoder_hidden_dim = 1024,      # 解码器隐藏层维度
                decoder_num_layer = 1,          # 解码器GRU层数
                bos_idx = 1, eos_idx = 3,       # 特殊token索引
                max_length=512                  # 最大生成长度
                ):
        super().__init__()
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.max_length = max_length
        
        # 负责将源语言句子编码为隐藏状态序列
        self.encoder = Encoder(src_vocab_size, embedding_dim=encoder_embedding_dim, hidden_dim=encoder_hidden_dim, num_layers=encoder_num_layer)
        # 负责在注意力机制辅助下生成目标语言句子
        self.decoder = Decoder(trg_vocab_size, embedding_dim=decoder_embedding_dim, hidden_dim=decoder_hidden_dim, num_layers=decoder_num_layer)
        
    def forward(self, *, encoder_inputs, decoder_inputs, attn_mask = None):
        """
        训练模式前向传播
        
        Args:
            encoder_inputs: 源语言输入序列，形状 [batch_size, src_seq_len]
                           例如: [[1, 4, 5, 3, 0, 0], ...] (西班牙语句子)
                           
            decoder_inputs: 目标语言输入序列，形状 [batch_size, trg_seq_len]
                           例如: [[1, 4, 5, 3, 0, 0], ...] (英语句子，包含[BOS])
                           
            attn_mask: 注意力掩码，形状 [batch_size, src_seq_len]
                      用于屏蔽源语言序列的padding位置，1表示padding位置
        
        Returns:
            logits: 所有时间步的预测logits，形状 [batch_size, trg_seq_len, trg_vocab_size]
                   每个时间步的预测结果，用于计算损失
                   
            scores: 所有时间步的注意力权重，形状 [batch_size, trg_seq_len, src_seq_len]
                   每个解码步对源语言序列的注意力分布，可用于可视化对齐关系
        """
        
        # ============ 编码阶段 ============
        # 将源语言句子输入编码器
        # encoder_outputs: [batch_size, src_seq_len, encoder_hidden_dim]
        # 编码器最后时间步的输出，用于注意力计算
        # encoder_last_hidden: [num_layers, batch_size, encoder_hidden_dim]
        encoder_outputs, encoder_last_hidden = self.encoder(encoder_inputs)
        batch_size, seq_len = decoder_inputs.shape
        
        # 存储每个解码步的输出
        logits_list = []
        scores_list = []
        
        # 初始化解码器隐藏状态为编码器的最后隐藏状态
        # 取最后一层（如果有多层GRU）作为解码器的初始隐藏状态
        # encoder_last_hidden[-1] 形状: [batch_size, hidden_dim]
        prev_hidden = encoder_last_hidden
        
        # ============ 解码阶段（逐时间步生成） ============
        for i in range(seq_len):
            # 当前时间步的解码器输入：目标序列的第i个词
            # decoder_inputs[:, i:i+1] 保持维度为 [batch_size, 1]
            # 这是教师强制：使用真实的目标词作为输入
            logits, prev_hidden, score = self.decoder(decoder_inputs[:, i:i+1], prev_hidden[-1], encoder_outputs, attn_mask=attn_mask)
            # logits.shape: [batch_size, 1, trg_vocab_size]
            # score.shape: [batch_size, src_seq_len, 1]
            
            logits_list.append(logits)  # 记录预测的logits，用于计算损失
            scores_list.append(score)  # 记录注意力分数,用于画图
        
        return torch.cat(logits_list, dim=-2), torch.cat(scores_list, dim=-1)
        
    @torch.no_grad() # 不计算梯度
    def infer(self, encoder_inputs, attn_mask = None):
        """
        推理模式：生成目标语言序列（不使用教师强制）
        
        Args:
            encoder_inputs: 源语言输入序列，形状 [1, src_seq_len]
                           推理时通常一次只处理一个句子
                           
            attn_mask: 注意力掩码，形状 [1, src_seq_len]
                      用于屏蔽源语言序列的padding位置
        
        Returns:
            preds_list: 生成的词索引列表，长度 <= max_length
                       例如: [1, 4, 5, 3] 表示 [BOS, hello, world, EOS]
                       
            scores: 所有时间步的注意力权重，形状 [trg_seq_len, src_seq_len]
                   显示翻译过程中的对齐关系
        """
        
        # ============ 编码阶段 ============
        # 将源语言句子输入编码器
        # encoder_input.shape = [1, sequence length]
        encoder_outputs, encoder_last_hidden = self.encoder(encoder_inputs)
        
        # ============ 初始化解码 ============
        # 初始化解码器输入为[BOS]标记
        # 形状: [1, 1]
        decoder_input = torch.Tensor([self.bos_idx]).reshape(1,1).to(dtype=torch.int64)
        decoder_pred = None
        
        # 存储生成的结果
        preds_list = []      # 存储生成的词索引
        score_list = []       # 存储每个时间步的注意力权重
        
        # ============ 逐步生成 ============
        # 初始化解码器隐藏状态为编码器的最后隐藏状态
        prev_hidden = encoder_last_hidden
        # 从开始标记 bos_idx 开始，迭代地生成序列，直到生成结束标记 eos_idx 或达到最大长度 max_length。
        for _ in range(self.max_length):
            # 单步解码
            # logits: [1, 1, trg_vocab_size] - 当前步的预测logits
            # prev_hidden: [1, hidden_dim] - 更新后的隐藏状态
            # score: [1, src_seq_len, 1] - 当前步的注意力权重
            logits, prev_hidden, score = self.decoder(decoder_input, prev_hidden[-1], encoder_outputs, attn_mask=attn_mask)
            decoder_pred = logits.argmax(dim=-1)    # 贪婪解码：选择概率最高的词作为当前步的预测，# decoder_pred.shape: [1, 1]
            decoder_input = decoder_pred            # 准备下一步的输入（使用当前步的预测）
            preds_list.append(decoder_pred.reshape(-1).item())      # 记录当前步的预测（转换为Python标量）
            score_list.append(score)                # 记录当前步的注意力权重（去除batch和最后一维）
            
            # 如果生成了[EOS]标记，提前停止生成
            if decoder_pred == self.eos_idx:
                break
        
        return preds_list, torch.cat(score_list, dim=-1)
    

# 2. Train Module

In [ ]:
def cross_entry_with_padding(logits, labels, padding_mask = None):
    """
    在序列到序列模型中，一个批次内的句子长度不同，需要用[PAD]填充到相同长度。
    这些填充位置不应该参与损失计算，否则会影响模型训练。
    """
    # logits.shape = [batch size, seq_len, num of classes]
    # labels.shape = [batch size, seq_len]
    # padding_mask = [batch size, seq_len]
    batch_size, seq_len, classes = logits.shape
    
    # 将logits从 [batch_size, seq_len, classes] 重塑为 [batch_size * seq_len, classes]
    # 这样每个时间步的预测变成一个独立的样本
    logits_flat = logits.reshape(batch_size * seq_len, classes)
    # 将labels从 [batch_size, seq_len] 重塑为 [batch_size * seq_len]
    # 每个时间步的真实标签对应一个预测
    labels_flat = labels.reshape(-1)
    # 计算每个位置的损失
    loss = F.cross_entropy(logits_flat, labels_flat, reduction='none') # reduction='none' 表示不对批次求平均，返回每个样本的损失,返回形状: [batch_size * seq_len]
    
    if padding_mask is None:
        loss = loss.mean()  # 如果没有掩码，简单地对所有位置求平均
    else:
        padding_mask = 1 - padding_mask.reshape(-1)
        loss = torch.mul(loss, padding_mask).sum() / padding_mask.sum()
        
    return loss

In [ ]:

class TensorBoardCallback:
    def __init__(self, log_dir, flush_secs=10):
        """
        Args:
            log_dir (str): dir to write log.
            flush_secs (int, optional): write to dsk each flush_secs seconds. Defaults to 10.
        """
        self.writer = SummaryWriter(log_dir=log_dir, flush_secs=flush_secs)

    def draw_model(self, model, input_shape):
        self.writer.add_graph(model, input_to_model=torch.randn(input_shape))

    def add_loss_scalars(self, step, loss, val_loss):
        self.writer.add_scalars(
            main_tag="training/loss",
            tag_scalar_dict={"loss": loss, "val_loss": val_loss},
            global_step=step,
        )

    def add_acc_scalars(self, step, acc, val_acc):
        self.writer.add_scalars(
            main_tag="training/accuracy",
            tag_scalar_dict={"accuracy": acc, "val_accuracy": val_acc},
            global_step=step,
        )

    def add_lr_scalars(self, step, learning_rate):
        self.writer.add_scalars(
            main_tag="training/learning_rate",
            tag_scalar_dict={"learning_rate": learning_rate},
            global_step=step,

        )

    def __call__(self, step, **kwargs):
        # add loss
        loss = kwargs.pop("loss", None)
        val_loss = kwargs.pop("val_loss", None)
        if loss is not None and val_loss is not None:
            self.add_loss_scalars(step, loss, val_loss)
        # add acc
        acc = kwargs.pop("acc", None)
        val_acc = kwargs.pop("val_acc", None)
        if acc is not None and val_acc is not None:
            self.add_acc_scalars(step, acc, val_acc)
        # add lr
        learning_rate = kwargs.pop("lr", None)
        if learning_rate is not None:
            self.add_lr_scalars(step, learning_rate)


class SaveCheckpointsCallback:
    def __init__(self, save_dir, save_step=5000, save_best_only=True):
        """
        Save checkpoints each save_epoch epoch.
        We save checkpoint by epoch in this implementation.
        Usually, training scripts with pytorch evaluating model and save checkpoint by step.

        Args:
            save_dir (str): dir to save checkpoint
            save_epoch (int, optional): the frequency to save checkpoint. Defaults to 1.
            save_best_only (bool, optional): If True, only save the best model or save each model at every epoch.
        """
        self.save_dir = save_dir
        self.save_step = save_step
        self.save_best_only = save_best_only
        self.best_metrics = - np.inf

        # mkdir
        if not os.path.exists(self.save_dir):
            os.mkdir(self.save_dir)

    def __call__(self, step, state_dict, metric=None):
        if step % self.save_step > 0:
            return

        if self.save_best_only:
            assert metric is not None
            if metric >= self.best_metrics:
                # save checkpoints
                torch.save(state_dict, os.path.join(self.save_dir, "best.ckpt"))
                # update best metrics
                self.best_metrics = metric
        else:
            torch.save(state_dict, os.path.join(self.save_dir, f"{step}.ckpt"))


class EarlyStopCallback:
    def __init__(self, patience=5, min_delta=0.01):
        """

        Args:
            patience (int, optional): Number of epochs with no improvement after which training will be stopped.. Defaults to 5.
            min_delta (float, optional): Minimum change in the monitored quantity to qualify as an improvement, i.e. an absolute
                change of less than min_delta, will count as no improvement. Defaults to 0.01.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.best_metric = - np.inf
        self.counter = 0

    def __call__(self, metric):
        if metric >= self.best_metric + self.min_delta:
            # update best metric
            self.best_metric = metric
            # reset counter
            self.counter = 0
        else:
            self.counter += 1

    @property
    def early_stop(self):
        return self.counter >= self.patience

In [ ]:
@torch.no_grad()
def evaluating(model, dataloader, loss_func):
    """
    模型评估函数：在验证集/测试集上计算模型的平均损失
    
    这个函数用于评估训练好的模型在未见过的数据上的表现。
    使用 @torch.no_grad() 装饰器禁用梯度计算，提高评估速度并节省内存。
    """
    loss_list = []
    for batch in dataloader:
        encoder_inputs = batch["encoder_inputs"]
        encoder_inputs_mask = batch["encoder_inputs_mask"]
        decoder_inputs = batch["decoder_inputs"]
        decoder_labels = batch["decoder_labels"]
        decoder_labels_mask = batch["decoder_labels_mask"]
        
        logits, _ = model(encoder_inputs=encoder_inputs, decoder_inputs=decoder_inputs, attn_mask=encoder_inputs_mask)
        loss = loss_func(logits, decoder_labels, padding_mask=decoder_labels_mask)
        loss_list.append(loss.cpu().item()) 
    
    return np.mean(loss_list)
    

In [ ]:
def training(model, train_dataloader, val_dataloader, epoch, loss_func, optimizer, 
             eval_step = 500, tensorboard_callback = None, save_ckpt_callback = None, early_stop_callback = None):
    record_dict = {
        "train" : [],
        "val" : []
    }
    
    global_step = 1
    
    model.train()
    with tqdm(total = epoch * len(train_dataloader)) as pbar:
        for epoch_id in range(epoch):
            # training
            for batch in train_dataloader:
                encoder_inputs = batch["encoder_inputs"]
                encoder_inputs_mask = batch["encoder_inputs_mask"]
                decoder_inputs = batch["decoder_inputs"]
                decoder_labels = batch["decoder_labels"]
                decoder_labels_mask = batch["decoder_labels_mask"]
            
                # 梯度清空
                optimizer.zero_grad()
                
                # 前向计算
                logits, _ = model(encoder_inputs=encoder_inputs, decoder_inputs=decoder_inputs, attn_mask=encoder_inputs_mask)
                loss = loss_func(logits, decoder_labels, padding_mask=decoder_labels_mask)
                
                # 梯度回传
                loss.backward()
                
                # 调整优化器，包括学习率的变动等
                optimizer.step()
    
                loss = loss.cpu().item()
                
                # 记录训练损失和步数
                record_dict['train'].append({'loss' : loss, 'step' : global_step})
                
                if global_step % eval_step == 0:
                    model.eval()
                    val_loss = evaluating(model, val_dataloader, loss_func)
                    record_dict['val'].append({'loss' : val_loss, 'step' : global_step})
                    model.train()
                    
                    # 1. 使用 tensorboard 可视化
                    if tensorboard_callback is not None:
                        tensorboard_callback(
                            global_step,
                            loss=loss, val_loss=val_loss,
                            lr=optimizer.param_groups[0]["lr"],
                            )
    
                    # 2. 保存模型权重 save model checkpoint
                    if save_ckpt_callback is not None:
                        save_ckpt_callback(global_step, model.state_dict(), metric=-val_loss)
    
                    # 3. 早停 Early Stop
                    if early_stop_callback is not None:
                        early_stop_callback(-val_loss)
                        if early_stop_callback.early_stop:
                            print(f"Early stop at epoch {epoch_id} / global_step {global_step}")
                            return record_dict
                
                # 更新步数
                global_step += 1
                pbar.update(1)
            pbar.set_postfix({"epoch": epoch_id, "loss": loss, "val_loss": val_loss}) # 更新进度条
        
    return record_dict

# 3. Prepare Train

In [ ]:
epoch = 20
batch_size = 64

train_ds = LangPairDataset("train")
test_ds = LangPairDataset("test")

src_word2idx, src_idx2word = get_word_idx(train_ds, "src") #源语言词表
trg_word2idx, trg_idx2word = get_word_idx(train_ds, "trg") #目标语言词表

src_tokenizer = Tokenizer(word2idx=src_word2idx, idx2word=src_idx2word) #源语言tokenizer
trg_tokenizer = Tokenizer(word2idx=trg_word2idx, idx2word=trg_idx2word) #目标语言tokenizer

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fct)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fct)

model = Sequence2Sequence(trg_vocab_size=len(trg_word2idx),src_vocab_size=len(src_idx2word))
print('模型总参数:', sum(i[1].numel() for i in model.named_parameters()))
loss_func = cross_entry_with_padding
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

exp_name = "translate-seq2seq"
if not os.path.exists("runs"):
    os.mkdir("runs")
tensorboard_callback = TensorBoardCallback(f"runs/{exp_name}")
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")
save_ckpt_callback = SaveCheckpointsCallback(f"checkpoints/{exp_name}", save_step=200, save_best_only=True)
early_stop_callback = EarlyStopCallback(patience=10, min_delta=0.001)

# 4. Start Train

In [ ]:
model = model.to(device)
record = training(
    model,
    train_dl,
    test_dl,
    epoch,
    loss_func,
    optimizer,
    tensorboard_callback=None,
    save_ckpt_callback=save_ckpt_callback,
    early_stop_callback=early_stop_callback,
    eval_step=200
    )

# 5. Plot Training Curve

In [ ]:
def plot_training_curves(record, save_path=None, figsize=(12, 6)):
    """
    绘制训练和验证损失曲线（美化版）
    
    Args:
        record: 包含训练和验证记录的字典
               格式: {"train": [{"step": step, "loss": loss}, ...],
                     "val": [{"step": step, "loss": loss}, ...]}
        save_path: 如果提供，保存图片到指定路径
        figsize: 图表大小
    """
    
    # # 设置中文字体（如果有中文需求）
    # plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
    # plt.rcParams['axes.unicode_minus'] = False    # 用来正常显示负号
    
    # 创建画布
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    
    # 提取数据
    train_steps = [i["step"] for i in record["train"]]
    train_losses = [i["loss"] for i in record["train"]]
    val_steps = [i["step"] for i in record["val"]]
    val_losses = [i["loss"] for i in record["val"]]
    
    # 绘制训练损失（细线，浅色）
    ax.plot(train_steps, train_losses, 
            label='Train Loss', 
            color='#1f77b4',           
            linewidth=1.5, 
            alpha=0.7,                  
            linestyle='-')
    
    # 绘制验证损失（粗线，深色，带标记）
    ax.plot(val_steps, val_losses, 
            label='Validation Loss', 
            color='#d62728',           
            linewidth=2.5, 
            marker='o',                 
            markersize=4,
            markevery=5,                
            markerfacecolor='white',    
            markeredgewidth=1.5,
            markeredgecolor='#d62728')
    
    # 找到最佳验证损失点
    best_val_idx = np.argmin(val_losses)
    best_val_step = val_steps[best_val_idx]
    best_val_loss = val_losses[best_val_idx]
    
    # 标注最佳验证损失点
    ax.scatter(best_val_step, best_val_loss, 
               color='green', s=100, zorder=5,
               marker='*', edgecolors='black', linewidth=1,
               label=f'Best Val Loss: {best_val_loss:.4f}')
    
    # 添加水平线表示最佳损失
    ax.axhline(y=best_val_loss, color='gray', linestyle='--', 
               linewidth=1, alpha=0.5)
    
    # 设置标题和标签
    ax.set_title('Training and Validation Loss Curves', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Training Steps', fontsize=12, fontweight='semibold')
    ax.set_ylabel('Loss Value', fontsize=12, fontweight='semibold')
    
    # 设置网格
    ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)
    ax.set_axisbelow(True)  # 网格在数据下方
    
    # 设置图例
    ax.legend(loc='upper right', fontsize=10, framealpha=0.95,
              shadow=True, fancybox=True, edgecolor='black')
    
    # 设置坐标轴范围（可选，可以根据数据自动调整）
    ax.set_xlim(min(train_steps), max(train_steps))
    ax.set_ylim(0, max(train_losses) * 1.1)  # y轴从0开始，留10%空间
    
    # 添加次要网格
    ax.minorticks_on()
    ax.grid(True, which='minor', linestyle=':', alpha=0.2)
    
    # 设置坐标轴刻度
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.tick_params(axis='both', which='minor', labelsize=8)
    
    # 添加背景色
    ax.set_facecolor('#f8f9fa')
    fig.patch.set_facecolor('white')
    
    # 添加文本注释（显示最终损失）
    final_train_loss = train_losses[-1]
    final_val_loss = val_losses[-1]
    text_str = f'Final Train Loss: {final_train_loss:.4f}\nFinal Val Loss: {final_val_loss:.4f}'
    ax.text(0.02, 0.98, text_str, transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 调整布局
    plt.tight_layout()
    
    # 保存图片（如果指定了路径）
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', 
                   facecolor='white', edgecolor='none')
        print(f"Figure saved to: {save_path}")
    
    # 显示图表
    plt.show()
    
    return fig, ax


# ============ 更简洁的版本（快速绘图） ============

def quick_plot_losses(record):
    """
    快速绘制损失曲线（简洁版）
    """
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    
    train_steps = [i["step"] for i in record["train"]]
    train_losses = [i["loss"] for i in record["train"]]
    val_steps = [i["step"] for i in record["val"]]
    val_losses = [i["loss"] for i in record["val"]]
    
    ax.plot(train_steps, train_losses, 'b-', label='Train', linewidth=1, alpha=0.7)
    ax.plot(val_steps, val_losses, 'r-o', label='Validation', linewidth=2, markersize=3)
    
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()


# ============ 带平滑效果的版本 ============

def smooth_curve(points, factor=0.8):
    """
    使用指数移动平均平滑曲线
    
    Args:
        points: 原始数据点
        factor: 平滑因子（0-1），越大越平滑
    
    Returns:
        平滑后的数据点
    """
    smoothed = []
    for point in points:
        if smoothed:
            previous = smoothed[-1]
            smoothed.append(previous * factor + point * (1 - factor))
        else:
            smoothed.append(point)
    return smoothed


def plot_smoothed_losses(record, save_path=None):
    """
    绘制平滑后的损失曲线
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    train_steps = [i["step"] for i in record["train"]]
    train_losses = [i["loss"] for i in record["train"]]
    val_steps = [i["step"] for i in record["val"]]
    val_losses = [i["loss"] for i in record["val"]]
    
    # 原始曲线
    ax1.plot(train_steps, train_losses, 'b-', alpha=0.3, linewidth=1, label='Train (raw)')
    ax1.plot(val_steps, val_losses, 'r-', alpha=0.3, linewidth=1, label='Val (raw)')
    
    # 平滑曲线
    smooth_train = smooth_curve(train_losses)
    smooth_val = smooth_curve(val_losses)
    
    ax1.plot(train_steps, smooth_train, 'b-', linewidth=2, label='Train (smoothed)')
    ax1.plot(val_steps, smooth_val, 'r-', linewidth=2, label='Val (smoothed)')
    
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('Raw vs Smoothed Loss Curves')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # 对数坐标
    ax2.semilogy(train_steps, train_losses, 'b-', alpha=0.5, label='Train')
    ax2.semilogy(val_steps, val_losses, 'r-o', label='Validation', markersize=3)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Loss (log scale)')
    ax2.set_title('Loss Curves (Log Scale)')
    ax2.grid(True, alpha=0.3, which='both')
    ax2.legend()
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

# 6. Infer

In [ ]:
# 加载最好的模型
model = Sequence2Sequence(len(trg_word2idx), len(src_word2idx))
model.load_state_dict(torch.load(f"./checkpoints/translate-seq2seq/best.ckpt", map_location="cpu"))

In [ ]:
class Translator:
    def __init__(self, model, src_tokenizer, trg_tokenizer):
        self.model = model
        self.model.eval() # 切换到验证模式
        self.src_tokenizer = src_tokenizer
        self.trg_tokenizer = trg_tokenizer
        
    def __call__(self, src_sentence):
        src_sentence = preprocess_sentence(src_sentence)
        encoder_inputs, attn_mask = self.src_tokenizer.encode([src_sentence.split()], 
                                                              padding_first=True, 
                                                              add_bos=True, 
                                                              add_eos=True, 
                                                              return_mask=True) # 对输入进行编码，并返回encode_piadding_mask
        
        encoder_inputs = torch.Tensor(encoder_inputs).to(dtype=torch.int64)
        
        preds, scores = self.model.infer(encoder_inputs=encoder_inputs, attn_mask=attn_mask)
        
        trg_sentence = self.trg_tokenizer.decode([preds], split=True, remove_eos=False)[0]
        
        return " ".join(trg_sentence[:-1])

In [ ]:
def evaluate_bleu_on_test_set(test_data, translator, print_head10 = False):
    """
    在测试集上计算平均 BLEU 分数。
    :param test_data: 测试集数据，格式为 [(src_sentence, [ref_translation1, ref_translation2, ...]), ...]
    :param translator: 翻译器对象（Translator 类的实例）
    :return: 平均 BLEU 分数
    """
    total_bleu = 0.0
    num_samples = len(test_data)
    
    for src_sentence, ref_translations in test_data:
        # 使用翻译器生成翻译结果
        candidate_translation = translator(src_sentence)

        # 计算 BLEU 分数
        bleu_score = sentence_bleu([ref_translations.split()], candidate_translation.split(),weights=(1, 0, 0, 0))
        total_bleu += bleu_score
        
        if print_head10 == True:
            # 打印当前句子的 BLEU 分数（可选）
            i=0
            print(f"Source: {src_sentence}")
            print(f"Reference: {ref_translations}")
            print(f"Candidate: {candidate_translation}")
            print(f"BLEU: {bleu_score:.4f}")
            print("-" * 50)
            i+=1
            if i>10:
                break
        
    # 计算平均 BLEU 分数
    avg_bleu = total_bleu / num_samples
    return avg_bleu